# Lithium supply chain optimisation - Energies 2024

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/lithium-optsc-energies-2024/blob/main/notebooks/00_walkthrough.ipynb)

The model behind [Jones (2024), *Energies* 17, 2685](https://doi.org/10.3390/en17112685).

**This notebook is thin on purpose.** It imports the package and calls it; it contains no model
logic, so it cannot drift from the code that produced the paper. To read the model, read
`src/lithium_energies/model.py`.


## 1. Install

On Colab, install the package from the repository. Locally, `pip install -e ".[dev]"` once.

In [ ]:
# Colab: clone the repository, then install FROM that checkout.
#
# `pip install git+https://...` is not enough, and fails in a way that looks
# like a bug in the model. It installs the package but not `data/`, which is
# not part of the wheel - and the installed `paths.py` then derives the repo
# root relative to site-packages, so the first cell that loads data dies with a
# FileNotFoundError naming a directory inside the interpreter. Cloning keeps the
# code and the data together in the layout `paths.py` expects.
import os
import subprocess
import sys

REPO = "lithium-optsc-energies-2024"

if "google.colab" in sys.modules:
    if os.path.basename(os.getcwd()) != REPO:
        if not os.path.isdir(REPO):
            subprocess.run(
                ["git", "clone", "-q",
                 "https://github.com/sear-labs/" + REPO + ".git"], check=True)
        os.chdir(REPO)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."],
                   check=True)
    print("cloned and installed", REPO, "into", os.getcwd())
else:
    print("local environment - assuming `pip install -e \".[dev]\"` has been run")


## 2. Gurobi licence — and what runs without one

**Most readers need nothing here.** The published objective can be verified in section 4
with **no solver and no licence at all**, because the repository ships the frozen model
instance (`.mps`) and its solution (`.sol`). That verification is this repository's main
claim, and it is the part that runs everywhere.

A licence is needed only to *re-solve* the model in section 5. The model is far larger than
the free `pip` licence allows (~2,000 variables; this one has 10,706 continuous variables
and 13,556 constraint rows), so the free licence cannot do it.

| | what it is | works in Colab? |
|---|---|---|
| **Named-User Academic** | a `gurobi.lic` file | **No** — node-locked to one machine, and the Colab VM is a different machine every session |
| **WLS Academic** | three values checked over the network | **Yes** — this is the one you want |

Both are free for academics from the Gurobi User Portal. Put the three values into Colab's
**Secrets** (key icon, left sidebar) as `GRB_WLSACCESSID`, `GRB_WLSSECRET`, `GRB_LICENSEID`,
or set the same three names as environment variables when running locally.

A Colab secret binds to your Google account, not to the notebook — so this notebook can be
shared or forked and still carries no key.

**Finding three values is not the same as them working.** The next cell treats those as two
separate questions, and never stops the notebook either way.


In [ ]:
import os

import gurobipy as gp

# Environment first, Colab secrets second, so the same notebook serves both.
# Never a literal: a key committed to a repository is exposed the moment the
# repository is shared, and deleting it later does not remove it from history.
KEYS = ("WLSACCESSID", "WLSSECRET", "LICENSEID")
found = {k: os.environ.get("GRB_" + k) for k in KEYS}
if not all(found.values()):
    try:
        from google.colab import userdata

        found = {k: userdata.get("GRB_" + k) for k in KEYS}
    except Exception:
        found = {}

ENV = None          # gp.Model(env=None) means "use the default licence"
if found and all(found.values()):
    try:
        ENV = gp.Env(params={"WLSACCESSID": found["WLSACCESSID"],
                             "WLSSECRET": found["WLSSECRET"],
                             # int(), not the raw value: Colab's userdata.get()
                             # returns strings and a string LICENSEID is rejected
                             # without saying why.
                             "LICENSEID": int(found["LICENSEID"])})
    except (gp.GurobiError, ValueError) as err:
        # Finding the three values does not mean they work. A rotated secret, an
        # expired licence or one typo'd character all land here. Report it and
        # carry on - section 4 does not need a licence.
        print("WLS credentials found, but Gurobi rejected them:", err)
        print("Check for a rotated secret, an expiry, or a typo.\n")

HAVE_WLS = ENV is not None
print("WLS licence:", "ready" if HAVE_WLS else "not configured")
print("Section 4 (verification, no solver) runs either way.")
print("Section 5 (re-solve) needs the licence.")


## 3. Build the instance and check it against the paper

The paper states the model had **13,556 rows** and **10,706 continuous variables**. Rebuilding from
`data/raw/` should reproduce both exactly - that is the check that the data pipeline still produces
the published instance.

In [ ]:
import pathlib
import subprocess
import sys

import lithium_energies

# Derived from the installed package, NOT from the working directory. A relative
# "../scripts/..." breaks the moment the notebook is run from anywhere but its
# own folder - which is exactly what happens on Colab after the clone.
ROOT = pathlib.Path(lithium_energies.__file__).resolve().parents[2]


def run_script(name, *args):
    """Run a repo script and show everything it said.

    `capture_output=True` while printing only .stdout hides stderr, so a failing
    script prints an empty line and looks like it did nothing. Show both, and the
    return code.
    """
    r = subprocess.run([sys.executable, str(ROOT / "scripts" / name), *args],
                       capture_output=True, text=True)
    print(r.stdout, end="")
    if r.stderr.strip():
        print("--- stderr ---")
        print(r.stderr, end="")
    if r.returncode != 0:
        print("--- exited %d ---" % r.returncode)
    return r


run_script("run_all.py")


## 4. Verify the published objective — no solver, no licence

The paper's objective is an *incumbent*, not a proven optimum: the solve stopped on a time
limit with a residual gap. So the repository ships the frozen instance and its solution, and
this recomputes the objective from them and checks every row, bound and integrality
condition by hand.

**This needs no Gurobi licence and no solver.** It is the claim that matters — that the
published number is reproducible — and anyone can check it.


In [ ]:
run_script("verify_solution.py")


## 5. Re-solve the model — needs the WLS licence

Optional. Everything above has already established that the published objective is correct;
this re-derives it from scratch. Ten-minute time limit, so it stops well short of the
paper's own run.


In [ ]:
if HAVE_WLS:
    run_script("run_all.py", "--solve", "600")
else:
    print("Skipped: no WLS licence configured.")
    print("Section 4 above already verified the published objective without one.")
    print("To re-solve, add GRB_WLSACCESSID / GRB_WLSSECRET / GRB_LICENSEID and re-run.")


## 5. What to cite

Cite **the paper** for the work and **this repository** for the code. `CITATION.cff` in the repo
root carries both, and GitHub renders a "Cite this repository" button from it.